## 05 — EDA: Single-Bidder Classifier

Exploratory analysis on contract award notices from TED. Builds and persists the train/test feature tables that the model notebook reads directly.

**Input:** `workspace.silver.contract_award_notices_subtype`  
**Output:** `workspace.gold.ml_features_train` / `workspace.gold.ml_features_test`

Covers: subtype and result-code filtering, FX conversion to EUR, target construction (`is_single_bidder`), stratified train/test split, categorical cleaning (small-category grouping learned on train only), log transform and median imputation for award value, and missingness flagging. All distribution-based decisions are made on the train set before any test data is touched.

In [0]:
%sql
-- Goal: list all currencies present in the training data (whitelisted awards only).
-- Tells us how many FX rates we need and whether EUR dominates.
SELECT
  currency,
  COUNT(*) AS rows,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM workspace.silver.contract_award_notices_subtype
WHERE notice_subtype IN ('29','30','31','32')
-- whitelisted subtypes are the actual award notices
GROUP BY currency
ORDER BY rows DESC

15% don't have currency, also don't have award value?

In [0]:
%sql
-- Goal: understand the null-currency rows. Do they even have a value?
-- If value is also null, currency doesn't matter. If value exists, we must decide a fallback.
SELECT
  CASE WHEN award_value IS NULL OR trim(award_value) = '' THEN 'no_value' ELSE 'has_value' END AS value_status,
  COUNT(*) AS rows
FROM workspace.silver.contract_award_notices_subtype
WHERE notice_subtype IN ('29','30','31','32')
  AND currency IS NULL
GROUP BY value_status

In [0]:
df = spark.table("workspace.silver.contract_award_notices_subtype")

In [0]:
# Goal: build the clean training base. Keep only real competitive awards:
#   - notice_subtype in 29-32 (standard awards, drops modifications/ex-ante/etc.)
#   - result_code = 'selec-w' (a winner was actually selected)
#   - award_value and num_tenders present (we need both the value feature and the target)
from pyspark.sql import functions as F

df = spark.table("workspace.silver.contract_award_notices_subtype")

df = df.filter(
    (F.col("notice_subtype").isin("29", "30", "31", "32")) &
    (F.col("result_code") == "selec-w") &
    (F.col("award_value").isNotNull()) & (F.trim(F.col("award_value")) != "") &
    (F.col("num_tenders").isNotNull()) & (F.trim(F.col("num_tenders")) != "")
)

print("Rows after filter:", df.count())

In [0]:
# Goal: inspect column types on the filtered base.
# We expect most columns as string (parser keeps raw text); confirms what we must cast to numeric.
df.printSchema()

In [0]:
# Goal: cast the columns that must be numeric, and convert award_value to EUR.
# award_value and num_tenders come as strings; the model and the EDA need them numeric.
# Categorical columns (country, cpv, procedure, buyer_type) stay as string on purpose.
from pyspark.sql import functions as F

# 1. Cast the raw numerics
df = df.withColumn("award_value", F.col("award_value").cast("double"))
df = df.withColumn("num_tenders", F.col("num_tenders").cast("int"))

# 2. Convert award_value to EUR using fixed approximate rates (size proxy, not accounting)
fx_to_eur = {
    "EUR": 1.0, "PLN": 4.3, "RON": 5.0, "CZK": 25.0, "SEK": 11.0,
    "CHF": 0.95, "HUF": 395.0, "DKK": 7.45, "NOK": 11.5, "GBP": 0.85,
    "USD": 1.08, "ISK": 150.0, "USN": 1.08, "UAH": 45.0, "RSD": 117.0, "MKD": 61.5,
}
eur_expr = F.lit(None).cast("double")
for code, rate in fx_to_eur.items():
    eur_expr = F.when(F.col("currency") == code, F.col("award_value") / F.lit(rate)).otherwise(eur_expr)
df = df.withColumn("award_value_eur", eur_expr)

# 3. Quick check that casts and conversion worked
df.select("award_value", "currency", "award_value_eur", "num_tenders").show(8, truncate=False)

In [0]:
# Goal: understand the target before splitting.
# (1) distribution of num_tenders, (2) class balance for the binary single-bidder label.
from pyspark.sql import functions as F

# Binary label: 1 if the typical awarded lot had a single bidder
df = df.withColumn("is_single_bidder", (F.col("num_tenders") == 1).cast("int"))

# (1) Raw distribution of num_tenders (top values)
print("--- num_tenders distribution ---")
df.groupBy("num_tenders").count().orderBy("num_tenders").show(20)

# (2) Class balance
print("--- single-bidder class balance ---")
df.groupBy("is_single_bidder").count().orderBy("is_single_bidder").show()

In [0]:
# Goal: split into train/test BEFORE any distribution-based decisions (prevents leakage).
# Stratify by the target so both sets keep the ~45/55 single-bidder balance.
# Spark's randomSplit doesn't stratify, so we split within each class and union.
train_frac = 0.8
seed = 42

class0 = df.filter(F.col("is_single_bidder") == 0)
class1 = df.filter(F.col("is_single_bidder") == 1)

train0, test0 = class0.randomSplit([train_frac, 1 - train_frac], seed=seed)
train1, test1 = class1.randomSplit([train_frac, 1 - train_frac], seed=seed)

train_df = train0.union(train1)
test_df  = test0.union(test1)

# Verify sizes and that the balance is preserved in both sets
print("Train rows:", train_df.count(), "| Test rows:", test_df.count())
print("--- train balance ---")
train_df.groupBy("is_single_bidder").count().orderBy("is_single_bidder").show()
print("--- test balance ---")
test_df.groupBy("is_single_bidder").count().orderBy("is_single_bidder").show()

In [0]:
# Goal: split into train/test BEFORE any distribution-based decisions (prevents leakage).
# Stratify by the target so both sets keep the ~45/55 single-bidder balance.
from pyspark.sql import functions as F

train_frac = 0.8
seed = 42

class0 = df.filter(F.col("is_single_bidder") == 0)
class1 = df.filter(F.col("is_single_bidder") == 1)

train0, test0 = class0.randomSplit([train_frac, 1 - train_frac], seed=seed)
train1, test1 = class1.randomSplit([train_frac, 1 - train_frac], seed=seed)

train_df = train0.union(train1)
test_df  = test0.union(test1)

print("Train rows:", train_df.count(), "| Test rows:", test_df.count())
print("--- train balance ---")
train_df.groupBy("is_single_bidder").count().orderBy("is_single_bidder").show()
print("--- test balance ---")
test_df.groupBy("is_single_bidder").count().orderBy("is_single_bidder").show()

In [0]:
# Goal: measure how strongly buyer_country relates to the target (on TRAIN only).
# Two things: (1) volume per country, (2) single-bidder rate per country.
# If the rate varies a lot across countries, country is a strong feature (and we must watch it doesn't dominate).
from pyspark.sql import functions as F

country_stats = (
    train_df.groupBy("buyer_country")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("is_single_bidder"), 3).alias("single_bidder_rate")
    )
    .orderBy(F.desc("rows"))
)
country_stats.show(30, truncate=False)

In [0]:
# Goal: understand the NULL-country rows (90% single-bidder is suspicious).
# Look at a few to decide: drop them, or keep as an explicit "unknown" category.
train_df.filter(F.col("buyer_country").isNull()) \
    .select("buyer_name", "cpv_code", "procedure_type", "award_value_eur", "num_tenders") \
    .show(10, truncate=False)

In [0]:
# Goal: count how many countries are small (<150 rows) to size an "OTHER" bucket.
country_stats.filter(F.col("rows") < 150).agg(
    F.count("*").alias("small_countries"),
    F.sum("rows").alias("rows_in_small_countries")
).show()

In [0]:
# Conclusion from previous: NULL-country rows are real buyers (mostly unparsed Italian
# ones), not junk, so we keep them as "UNKNOWN". 15 countries have <150 rows (1.2% of
# train) and get grouped into "OTHER" to avoid noisy tiny categories.
# Goal: clean buyer_country into a model-ready feature. Small-country list is learned
# from TRAIN only, then applied to both sets (prevents leakage).
from pyspark.sql import functions as F

small_countries = [
    r["buyer_country"] for r in
    country_stats.filter(F.col("rows") < 150).select("buyer_country").collect()
    if r["buyer_country"] is not None
]
print("Countries grouped into OTHER:", small_countries)

def clean_country(col):
    return (
        F.when(col.isNull(), F.lit("UNKNOWN"))
         .when(col.isin(small_countries), F.lit("OTHER"))
         .otherwise(col)
    )

train_df = train_df.withColumn("country_clean", clean_country(F.col("buyer_country")))
test_df  = test_df.withColumn("country_clean", clean_country(F.col("buyer_country")))

train_df.groupBy("country_clean").count().orderBy(F.desc("count")).show(40)

In [0]:
# Conclusion from previous: country is clean (26 categories, all with enough mass). OTHER
# correctly absorbed anomalies like Fiji/Curacao.
# Goal: derive cpv_division (first 2 digits = ~45 families) and measure the single-bidder
# rate per division on TRAIN, to assess whether technology CPVs show a distinct pattern.
from pyspark.sql import functions as F

train_df = train_df.withColumn("cpv_division", F.substring(F.col("cpv_code"), 1, 2))
test_df  = test_df.withColumn("cpv_division", F.substring(F.col("cpv_code"), 1, 2))

cpv_stats = (
    train_df.groupBy("cpv_division")
    .agg(
        F.count("*").alias("rows"),
        F.round(F.avg("is_single_bidder"), 3).alias("single_bidder_rate")
    )
    .orderBy(F.desc("rows"))
)
cpv_stats.show(50, truncate=False)

In [0]:
# Conclusion from previous: CPV division carries strong signal. Tech divisions differ from
# the mean and from each other (72=0.52, 48=0.59, 30=0.37 vs 0.45 overall), confirming CPV
# is a meaningful feature. Small divisions show unstable rates from tiny samples (e.g. 41=1.0 on 5 rows).
# Goal: group CPV divisions with <100 train rows into "OTHER" to remove noisy categories.
# Small-division list learned from TRAIN only, applied to both sets.
from pyspark.sql import functions as F

small_divisions = [
    r["cpv_division"] for r in
    cpv_stats.filter(F.col("rows") < 100).select("cpv_division").collect()
    if r["cpv_division"] is not None
]
print("CPV divisions grouped into OTHER:", small_divisions)

def clean_cpv(col):
    return (
        F.when(col.isNull() | (F.trim(col) == ""), F.lit("UNKNOWN"))
         .when(col.isin(small_divisions), F.lit("OTHER"))
         .otherwise(col)
    )

train_df = train_df.withColumn("cpv_division_clean", clean_cpv(F.col("cpv_division")))
test_df  = test_df.withColumn("cpv_division_clean", clean_cpv(F.col("cpv_division")))

train_df.groupBy("cpv_division_clean").count().orderBy(F.desc("count")).show(50)

In [0]:
# Conclusion from previous: CPV is clean (31 solid divisions + OTHER). Country and CPV are
# both strong features. Remaining categoricals to inspect: procedure_type and buyer_type.
# Goal: check categories, null rates and single-bidder rate per category for the two
# remaining categorical features, on TRAIN, to decide grouping or feature inclusion.
from pyspark.sql import functions as F

for c in ["procedure_type", "buyer_type"]:
    print(f"--- {c} ---")
    (train_df.groupBy(c)
        .agg(
            F.count("*").alias("rows"),
            F.round(F.avg("is_single_bidder"), 3).alias("single_bidder_rate")
        )
        .orderBy(F.desc("rows"))
        .show(30, truncate=False))

In [0]:
# Conclusion from previous: procedure_type carries a very strong, explainable signal
# (neg-wo-call = 0.87, by definition little competition; open = 0.43). buyer_type carries a
# milder but real signal (body-pl = 0.57 vs la = 0.38). Both have NULLs and small categories.
# Goal: group categories with <100 train rows into "OTHER" and NULLs into "UNKNOWN" for both
# features. Small-category lists learned from TRAIN only, applied to both sets.
from pyspark.sql import functions as F

def small_cats(col_name, threshold=100):
    stats = train_df.groupBy(col_name).count().filter(F.col("count") < threshold)
    return [r[col_name] for r in stats.select(col_name).collect() if r[col_name] is not None]

def clean_cat(col, small_list):
    return (
        F.when(col.isNull() | (F.trim(col) == ""), F.lit("UNKNOWN"))
         .when(col.isin(small_list), F.lit("OTHER"))
         .otherwise(col)
    )

small_proc = small_cats("procedure_type")
small_buyer = small_cats("buyer_type")
print("procedure_type -> OTHER:", small_proc)
print("buyer_type -> OTHER:", small_buyer)

train_df = train_df.withColumn("procedure_clean", clean_cat(F.col("procedure_type"), small_proc))
test_df  = test_df.withColumn("procedure_clean", clean_cat(F.col("procedure_type"), small_proc))
train_df = train_df.withColumn("buyer_type_clean", clean_cat(F.col("buyer_type"), small_buyer))
test_df  = test_df.withColumn("buyer_type_clean", clean_cat(F.col("buyer_type"), small_buyer))

print("--- procedure_clean ---")
train_df.groupBy("procedure_clean").count().orderBy(F.desc("count")).show()
print("--- buyer_type_clean ---")
train_df.groupBy("buyer_type_clean").count().orderBy(F.desc("count")).show()

In [0]:
# Conclusion from previous: all four categorical features are clean and carry signal
# (country, cpv_division, procedure, buyer_type). Last feature to examine is the numeric one:
# award_value_eur, which earlier samples suggested is extremely skewed (cents to millions).
# Goal: quantify the distribution of award_value_eur on TRAIN to decide whether a log
# transform is needed and whether extreme values need handling.
from pyspark.sql import functions as F

train_df.select(
    F.round(F.min("award_value_eur"), 2).alias("min"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.25)"), 2).alias("p25"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.5)"), 2).alias("median"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.75)"), 2).alias("p75"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.95)"), 2).alias("p95"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.99)"), 2).alias("p99"),
    F.round(F.max("award_value_eur"), 2).alias("max"),
    F.count(F.when(F.col("award_value_eur") <= 0, 1)).alias("zero_or_neg")
).show()

In [0]:
# Conclusion from previous: all four categorical features are clean and carry signal
# (country, cpv_division, procedure, buyer_type). Last feature to examine is the numeric one:
# award_value_eur, which earlier samples suggested is extremely skewed (cents to millions).
# Goal: quantify the distribution of award_value_eur on TRAIN to decide whether a log
# transform is needed and whether extreme values need handling.
from pyspark.sql import functions as F

train_df.select(
    F.round(F.min("award_value_eur"), 2).alias("min"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.25)"), 2).alias("p25"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.5)"), 2).alias("median"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.75)"), 2).alias("p75"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.95)"), 2).alias("p95"),
    F.round(F.expr("percentile_approx(award_value_eur, 0.99)"), 2).alias("p99"),
    F.round(F.max("award_value_eur"), 2).alias("max"),
    F.count(F.when(F.col("award_value_eur") <= 0, 1)).alias("zero_or_neg")
).show()

In [0]:
# Conclusion from previous: 1,103 rows have award_value <= 0 (incl. a -8,385 min), which the
# log can't handle. Before treating them as missing, it's worth checking what they are: a
# systematic parse issue might be recoverable rather than discarded (5% of train).
# Goal: inspect the zero/negative-value rows to find whether they share a cause (currency,
# country, procedure) or whether the original award_value string reveals a parsing problem.
from pyspark.sql import functions as F

neg = train_df.filter(F.col("award_value_eur") <= 0)

# How they split between exactly zero and truly negative
print("--- zero vs negative ---")
neg.groupBy(F.when(F.col("award_value_eur") == 0, "zero").otherwise("negative").alias("kind")) \
   .count().show()

# Do they cluster by currency or country?
print("--- by currency ---")
neg.groupBy("currency").count().orderBy(F.desc("count")).show(10)

# Look at the raw values alongside the converted ones
print("--- sample rows ---")
neg.select("award_value", "currency", "award_value_eur", "procedure_type", "buyer_country") \
   .show(15, truncate=False)

In [0]:
# Conclusion from previous: the 1,103 non-positive values are sentinels, not real amounts.
# Almost all are exactly -1.0 or 0.0, concentrated in German EUR notices (977 of 1,103),
# i.e. a publisher convention for "value not disclosed". Treating them as missing is correct;
# the rows are kept (they still have target and all categoricals), only the value becomes NULL.
# Goal: null out non-positive values, then create log_award_value = log of the cleaned value.
from pyspark.sql import functions as F

train_df = train_df.withColumn(
    "award_value_eur_clean",
    F.when(F.col("award_value_eur") > 0, F.col("award_value_eur")).otherwise(F.lit(None))
)
test_df = test_df.withColumn(
    "award_value_eur_clean",
    F.when(F.col("award_value_eur") > 0, F.col("award_value_eur")).otherwise(F.lit(None))
)

train_df = train_df.withColumn("log_award_value", F.log(F.col("award_value_eur_clean")))
test_df  = test_df.withColumn("log_award_value", F.log(F.col("award_value_eur_clean")))

train_df.select(
    F.round(F.min("log_award_value"), 2).alias("min"),
    F.round(F.expr("percentile_approx(log_award_value, 0.5)"), 2).alias("median"),
    F.round(F.max("log_award_value"), 2).alias("max"),
    F.count(F.when(F.col("log_award_value").isNull(), 1)).alias("nulls")
).show()

In [0]:
# Conclusion from previous: 1,103 rows have sentinel values (-1/0), mostly German EUR notices.
# Before imputing or flagging, it's worth ruling out a data-quality cause: wrong subtype,
# a single problematic publisher, or a parser miss (value present in XML but not extracted).
# Goal: profile the sentinel-value rows by subtype and source to identify a systematic cause.
from pyspark.sql import functions as F

neg = train_df.filter(F.col("award_value_eur") <= 0)

print("--- by notice_subtype ---")
neg.groupBy("notice_subtype").count().orderBy(F.desc("count")).show()

print("--- by buyer_country (all, not just top) ---")
neg.groupBy("buyer_country").count().orderBy(F.desc("count")).show()

print("--- a few source_file paths to inspect the raw XML ---")
neg.select("source_file").show(5, truncate=False)

In [0]:
# Conclusion from previous: sentinel-value rows are all valid subtypes (29-32), so not a type
# contamination. They span many countries (DEU heaviest at 840, but 260 elsewhere), pointing
# to a genuine non-disclosure convention rather than one publisher's bug. Decisive check left:
# is the value truly absent in the source XML, or present but missed by the parser?
# Goal: read one raw sentinel-value XML and look for any award value field.
raw = spark.read.text("dbfs:/Volumes/workspace/default/lakehouse/bronze/ted/2026/06/20260617/20260617_115/00416704_2026.xml", wholetext=True).first()["value"]
print(len(raw), "chars")

import re
# Show any tags that mention amount/value, to see if a real number exists anywhere
for m in re.findall(r'<[^>]*[Aa]mount[^>]*>[^<]*</[^>]*>', raw):
    print(m[:200])
print("---")
for m in re.findall(r'<[^>]*[Vv]alue[^>]*>[^<]*</[^>]*>', raw):
    print(m[:200])

In [0]:
# Conclusion from previous: raw XML confirms the value is genuinely absent at source
# (TotalAmount and PayableAmount both = -1, a non-disclosure sentinel), not a parser miss and
# not a wrong category. Rows are otherwise valid, so they're kept. Because the absence is
# structured (concentrated in DEU), a missingness flag lets the model use it as signal.
# Goal: create value_missing flag, then impute missing log_award_value with the TRAIN median.
# Median is computed on TRAIN only and applied to both sets (prevents leakage).
from pyspark.sql import functions as F

# 1. Missingness flag (1 = value was absent/sentinel, 0 = real value)
train_df = train_df.withColumn("value_missing", F.col("log_award_value").isNull().cast("int"))
test_df  = test_df.withColumn("value_missing", F.col("log_award_value").isNull().cast("int"))

# 2. Median of log_award_value from TRAIN only
median_log = train_df.approxQuantile("log_award_value", [0.5], 0.01)[0]
print("Train median log_award_value used for imputation:", round(median_log, 3))

# 3. Impute missing values with that median, in both sets
train_df = train_df.fillna({"log_award_value": median_log})
test_df  = test_df.fillna({"log_award_value": median_log})

# 4. Confirm no nulls remain and the flag counts make sense
train_df.select(
    F.count(F.when(F.col("log_award_value").isNull(), 1)).alias("remaining_nulls"),
    F.sum("value_missing").alias("flagged_missing")
).show()

In [0]:
# Conclusion from previous: log transform confirmed necessary. Median imputation done
# (12.206), 1,103 missingness flags set. This plot documents the transformation visually
# for the report.
# Goal: show the distribution of award_value_eur before and after log transform, side by
# side, to illustrate why the log is necessary for modeling.
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Sample to pandas (enough for a clean histogram)
sample = (
    train_df
    .filter("value_missing = 0")
    .select("award_value_eur_clean", "log_award_value")
    .limit(15000)
    .toPandas()
    .dropna()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#F8F9FA")
for ax in axes:
    ax.set_facecolor("#F8F9FA")
    ax.spines[["top", "right"]].set_visible(False)

# Left: raw value in EUR (log x-axis to show the range without flattening)
axes[0].hist(
    sample["award_value_eur_clean"],
    bins=80,
    color="#1A6EBD",
    edgecolor="none",
    alpha=0.85
)
axes[0].set_xscale("log")
axes[0].set_title("Raw award value (EUR)", fontsize=13, fontweight="bold", pad=10)
axes[0].set_xlabel("Value (log scale)", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"€{x:,.0f}"
))

# Right: log-transformed value
axes[1].hist(
    sample["log_award_value"],
    bins=80,
    color="#E87722",
    edgecolor="none",
    alpha=0.85
)
axes[1].set_title("Log-transformed award value", fontsize=13, fontweight="bold", pad=10)
axes[1].set_xlabel("ln(value)", fontsize=11)
axes[1].set_ylabel("")
axes[1].axvline(
    sample["log_award_value"].median(),
    color="#1A6EBD", linestyle="--", linewidth=1.4,
    label=f"Median = {sample['log_award_value'].median():.2f}"
)
axes[1].legend(fontsize=10)

plt.suptitle(
    "Award value distribution — before and after log transform",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("/tmp/award_value_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/award_value_distribution.png")

In [0]:
# Conclusion from previous: log distribution looks good (roughly bell-shaped, median 12.22).
# The raw value plot was unreadable because the 19bn outlier collapses all bins to zero.
# Goal: replot raw value capped at p99 on a linear scale to show the skew visibly,
# paired with the clean log distribution for the report.
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

p99 = sample["award_value_eur_clean"].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#F8F9FA")
for ax in axes:
    ax.set_facecolor("#F8F9FA")
    ax.spines[["top", "right"]].set_visible(False)

# Left: raw value capped at p99 so the shape is visible
axes[0].hist(
    sample["award_value_eur_clean"].clip(upper=p99),
    bins=80,
    color="#1A6EBD",
    edgecolor="none",
    alpha=0.85
)
axes[0].set_title("Raw award value (EUR, capped at p99)", fontsize=13, fontweight="bold", pad=10)
axes[0].set_xlabel("Value (EUR)", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"€{x/1e6:.1f}M" if x >= 1e6 else f"€{x/1e3:.0f}k"
))

# Right: log-transformed value
axes[1].hist(
    sample["log_award_value"],
    bins=80,
    color="#E87722",
    edgecolor="none",
    alpha=0.85
)
axes[1].set_title("Log-transformed award value", fontsize=13, fontweight="bold", pad=10)
axes[1].set_xlabel("ln(value)", fontsize=11)
axes[1].set_ylabel("")
axes[1].axvline(
    sample["log_award_value"].median(),
    color="#1A6EBD", linestyle="--", linewidth=1.4,
    label=f"Median = {sample['log_award_value'].median():.2f}"
)
axes[1].legend(fontsize=10)

plt.suptitle(
    "Award value distribution — before and after log transform",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("/tmp/award_value_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# Conclusion from previous: full EDA pipeline reconstructed after session reset.
# Goal: persist train and test feature tables to Delta so the model notebook
# can run independently without depending on this session staying alive.
from pyspark.sql import functions as F

CAT_FEATURES = [
    "country_clean",
    "cpv_division_clean", 
    "procedure_clean",
    "buyer_type_clean",
]
NUM_FEATURES = ["log_award_value", "value_missing"]
TARGET = "is_single_bidder"

cols = CAT_FEATURES + NUM_FEATURES + [TARGET]

train_df.select(cols).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.ml_features_train")

test_df.select(cols).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.ml_features_test")

print("Train saved:", spark.table("workspace.gold.ml_features_train").count(), "rows")
print("Test saved: ", spark.table("workspace.gold.ml_features_test").count(), "rows")